In [1]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import session
import h5py
import pickle

In [2]:
# set the main data directory (this needs to be changed by each user)
maindir = '/Volumes/ExternalSSD/Credit_assignement'

In [3]:
# get all the sessions to analyze
mouse = 'M1'
layers = 'L1' #'L23'
sessionfile = open('M1_L1_Sessions.txt','r') #Or change this for the given mouse and Layers
allsessions = [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

In [4]:
print(allsessions)

['720520553', '721038464', '722126561', '723323411']


In [ ]:
# create a dictionary with Session objects prepared for analysis
sdict = {}
for sess in allsessions:                       # remove the :1 to get all sessions ready
    print("\nCreating session {}...".format(sess))
    sdict[sess] = session.Session(maindir,sess)    # creates a session object to work with
    sdict[sess].extract_info()                     # extracts necessary info for analysis
    print("finished session {}.".format(sess))


Creating session 720520553...
Loading stimulus dictionary...
Loading alignment dataframe...
calculating stimulus alignment


In [ ]:
meansave = np.zeros([len(allsessions),4,8])
stdsave = np.zeros([len(allsessions),4,8])
for snum in range (0,len(allsessions)):
    S = allsessions[snum]
    print(S)
    
    #pull up h5 and df pkl files
    roifile = sdict[S].roi_traces
    twop = h5py.File(roifile,'r+')
    df_pkl_name = sdict[S].align_pkl
    file = open(df_pkl_name, 'rb')
    stim_p = pickle.load(file)
    file.close()
    stim_panda = stim_p['stim_df']

    #get df/f for Gabor frame "A","B","C","D", and for the whole Gabor Period
    for gbframe in range(0,4):
        gabor_segs = np.where((stim_panda['stimType'] == 'g') & (stim_panda['GABORFRAME'] == gbframe))
        gabor_starts = np.array(stim_panda['start_frame'][gabor_segs[0]])
        gabor_ends = np.array(stim_panda['end_frame'][gabor_segs[0]])

        meanroi = np.zeros([len(gabor_starts),8])
        for i in range(0,len(gabor_starts)):
            meanroi[i,:] = np.nanmean(twop['data'][:,gabor_starts.item(i):gabor_starts.item(i)+8],axis=0)
        
        meansave[snum,gbframe,:] = np.nanmean(meanroi,axis=0)
        stdsave[snum,gbframe,:] = np.nanstd(meanroi,axis=0)/np.sqrt(1.0*i)    
    

In [ ]:
for snum in range (0,len(allsessions)):
    plt.errorbar(np.linspace(0,0.3,8),meansave[snum,0,:],stdsave[snum,0,:],color='r')
    plt.hold(True)
    plt.errorbar(np.linspace(0,0.3,8),meansave[snum,1,:],stdsave[snum,1,:],color='k')
    plt.errorbar(np.linspace(0,0.3,8),meansave[snum,2,:],stdsave[snum,2,:],color='g')
    plt.errorbar(np.linspace(0,0.3,8),meansave[snum,3,:],stdsave[snum,3,:],color='b')
    plt.xlabel('time (s)')
    plt.ylabel('mean ROI df/f +/- SEM')
    plt.title('red:A, black:B, green:C, blue:D')
    plt.savefig('ROI_Analysis_'+mouse+'_Layer'+layers+'_GaborFrames_dff_SESS'+allsessions[snum],dpi=300)
    plt.hold(False)